## 1. Motivation.

The dataset used in this study is the Automated Traffic Volume Counts from NYC Open Data, covering traffic counts from 2018 to 2024.
It provides granular insights into traffic flow at the quarter-level, capturing vehicle counts on selected roads in NYC.
The dataset includes timestamps, road names, and directional traffic counts (northbound, eastbound, westbound, and southbound).
In this study, a file containing roads in NYC is also being used. The file is self-generated from a .shp layer, provided from the same NYC Open Data. The file includes the geometry as MultiLineStrings in the coordinate system EPSG:4326, for correct plotting.

I chose this dataset because i have a large interst in transportation in general. I found this dataset, with a lot of traffic counts located in NYC through 2018-2024. I was looking for data in Copenhagen, however the data was quite limited and i wasn't able to find anything for multiple years, covering enough locations for it to be representable. Therefore i opted to the NYC Open Data.

I initially wanted to show the general traffic patterns in NYC based on these automated traffic volume counts. It is important to note, that it is not possible to give a full perspective on the traffic patterns in NYC whith only traffic counts, as these are not collected in a scale, that represents every road, at any time. With this said, i thought it would be interesting to see, which dataanalyzis and visualizations one could retrieve from these data which i hoped mostly could be something like where, as in which roads are being used, and when, as in when on the day are people driving.

## 2. Basic stats

For the data cleaning and preprocessing I have filtered the Automated Traffic Volume Counts data, taking only data from 2018-2024. Otherwise, the dataset was very clean, and contained reasonable data for going further in the analyzis.

For the roadnetwork file, i only managed to find a .shp file with an incompatible geometry for reading in python. I therefore opened the file in another software caled QGIS, which handles many different projections and geometries. There, i created a new column holding the geometry of the roadsegments, and converted the geometry to a python compatible coordinate refrence system, EPSG:4326. This file holds The name of the street, a segmentID and the geometry of the roadsegment. Now i was able to join the two datasets on both street name and segmentID to retrieve the geometry of the different traffic counts for plotting.

The dataset consits of 528625 traffic volume counts across across all of NYC during 2018-2024. The dataset contains traffic measurements per 15-minute intervals, distributed across weekdays, times of the day, and NYC boroughs as well as driving direction. Not all counts appear consistent throughout the entire dataset, meaning that there are gaps of data collection on roads for som periods. These periods occur at random, and is usually due to technical problems from the counting devices on the roads.

Some of the key notes of the data is that weekdays show distinct rush hours in the morning (7-9 AM) and afternoon (3-5 PM), while weekends have flatter traffic patterns without strong peaks. Also that Tuesdays, Wednesdays, and Thursdays consistently have the highest average traffic volumes. And that Mondays, Fridays, and weekends experience reduced traffic, possibly due to remote work, extended weekends, or cultural habits (like resting on Sundays). Also some major roads, like Long Island Expressway and Grand Central Parkway, maintain consistently high traffic throughout the day, unlike others with clear rush hour peaks.

## 3. Data Analysis

This dataset provides a detailed look into traffic patterns across New York City’s five boroughs. To understand these patterns, I conducted exploratory data analysis (EDA), examining variations based on time, location, and direction.

I began by grouping data by quarter-hour intervals, allowing me to identify peak congestion periods and low-traffic hours. Next, I compared traffic trends across the five boroughs, revealing distinct differences in volume and flow. A bar plot offered an overview of average traffic per quarter-hour across all seven weekdays, while a line graph depicted borough-specific traffic fluctuations throughout an entire day.

To gain deeper insights, I created a subplot grid of nine graphs, where traffic volume was analyzed based on direction and roadway segments. Finally, a map visualization presented the traffic data geographically, using color coding, where red roads indicated heavy traffic, while yellow roads showed lighter traffic, offering an intuitive representation of NYC’s traffic hotspots.

## 4. Genre

## Genre of Data Story
This data story primarily falls under the annotated graph and map genres. The bar and line plots provide quantitative comparisons, showcasing traffic trends over time and across boroughs. The final map visualization introduces a spatial component, offering a clear representation of congestion hotspots in NYC. By combining these elements, the dataset tells a comprehensive story about traffic patterns in a way that is both analytical and visually intuitive.

## Tools Used in Visual Narrative Categories
The visualizations employ various techniques from Segal & Heer’s framework to enhance clarity and engagement:

### Visual Structuring

The consistent layout across all plots ensures intuitive comparisons.

The dropdown feature enables users to track traffic patterns dynamically.

### Highlighting

The map effectively uses color-coded roads to emphasize congestion levels.

Borough-specific plots distinguish traffic behavior across different regions.

### Transition Guidance

The dropdown function orders transitions smoothly, allowing viewers to switch between weekdays.

Motion in interactive elements ensures seamless exploration of data trends.

## Tools Used in Narrative Structure Categories
The data story also applies key elements of narrative structuring:

### Ordering

The sequence of plots moves logically from general trends (bar plot) to detailed street-level traffic behavior, guiding the reader through deeper insights.

### Interactivity

Hover tooltips and the dropdown selection enable users to explore borough-specific trends effortlessly.

Labels and annotations provide textual context, helping users understand fluctuations in traffic.

### Messaging

The synthesis of multiple graphs offers a complete, data-driven understanding of NYC’s traffic patterns.

The combination of different visual approaches ensures that the narrative remains engaging, informative, and easy to follow.

## 5. Visualizations

In [ ]:
import pandas as pd
import geopandas as gpd
import plotly.express as px
import geopandas as gpd
import branca.colormap as cm


# Importing df and gdf

df = pd.read_csv("df_reduced.csv")
gdf = gpd.read_file("merged_fixed.gpkg")


In [2]:
# Creating datetime columns

df_datetime = df[["Yr", "M", "D", "HH", "MM"]].copy()
df_datetime.columns = ["year", "month", "day", "hour", "minute"]

df["datetime"] = pd.to_datetime(df_datetime, errors="coerce")
df = df.dropna(subset=["datetime"])
df["hour"] = df["datetime"].dt.floor("h")

df["datetime"] = pd.to_datetime(df_datetime, errors="coerce")
df["date"] = df["datetime"].dt.date

df["weekday"] = df["datetime"].dt.day_name()



## Bar plots of average traffic per weekday:
Chosen to highlight differences in traffic volumes between weekdays and weekends, making it easy to compare the relative magnitude of traffic patterns

In [3]:
# Sorting weekdays in the correct order
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Calculating average traffic pr. weekday
avg_by_day = df.groupby("weekday")["Vol"].mean().reset_index()
avg_by_day["weekday"] = pd.Categorical(avg_by_day["weekday"], categories=weekday_order, ordered=True)
avg_by_day = avg_by_day.sort_values("weekday")

# Plotting
fig = px.bar(
    avg_by_day,
    x="weekday",
    y="Vol",
    title="Average traffic pr. quarter on weekdays",
    labels={"Traffic": "Average traffic pr. quarter", "weekday": "Weekday"},
    color_discrete_sequence=["lightblue"],
)

fig.update_layout(
    width=800,
    xaxis_title="Weekdays",
    yaxis_title="Average traffic pr. quarter",
    template="plotly_white"
)
# Vis grafen
fig.show()

## Line graphs of daily traffic patterns by borough:
These visualizations effectively show how traffic fluctuates over a 24-hour period in different parts of NYC. They clearly depict rush hour peaks and inter-borough differences

In [4]:
# Group by Boro, quarter, and weekday
df_grouped = df.groupby(["weekday", "Boro", "quarter"], as_index=False)["Vol"].mean()

# Group on borough and quarter and calculate the mean of Vol
df_grouped_avg = df.groupby(["Boro", "quarter"], as_index=False)["Vol"].mean()

# Plotting
fig = px.line(
    df_grouped_avg,
    x="quarter",
    y="Vol",
    color="Boro",
    labels={"Vol": "Average trafic", "quarter": "Quarter (0-95)", "Boro": "Borough"},
    title="Average trafic pr. quarter of the day"
)

fig.update_layout(
    xaxis_title="Quarter of the day",
    yaxis_title="Average trafic",
    title="Average trafic pr. quarter during the day"
)

fig.write_html("gennemsnitlig_trafik_bydele")

fig.show()


## Directional line plots for selected main roads:
These plots show traffic patterns in different directions (east, west, north, south) on major expressways. They are essential for understanding how traffic flows in/out of certain areas and how bidirectional traffic differs.

In [5]:
# Select roads
selected_streets = [
    "BROOKLYN BRIDGE",
    "LONG ISLAND EXPRESSWAY",
    "GRAND CENTRAL PARKWAY",
    "CROSS BRONX EXPRESSWAY",
    "BROOKLYN QUEENS EXPRESSWAY",
    "BELT PARKWAY",
    "VAN WYCK EXPRESSWAY",
    "BRONX RIVER PARKWAY",
    "KOSCIUSZKO BRIDGE"
]

df_selected = df[df["street"].isin(selected_streets)]

# Group pr. road, direction and quarter
df_grouped = df_selected.groupby(["street", "Direction", "quarter"], as_index=False)["Vol"].mean()

fig = px.line(
    df_grouped,
    x="quarter",
    y="Vol",
    color="Direction",
    facet_col="street",
    facet_col_wrap=3,  # Making 3 plots per row
    title="Traffic patterns for selected roads",
    labels={"quarter": "Quarter", "Vol": "Average Traffic", "Direction": "Direction"}
)

# Standardize X-axis labels as time format
fig.update_layout(
    height=1200,
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(0, 96, 8)),
        ticktext=[f"{(t*15)//60:02.0f}:{(t*15)%60:02.0f}" for t in range(0, 96, 8)]
    )
)

# Ensure all subplot x-axes match
fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(0, 96, 8)),  # Standardized ticks across all facets
    ticktext=[f"{(t*15)//60:02.0f}:{(t*15)%60:02.0f}" for t in range(0, 96, 8)],
    matches="x"  # Forces all subplots to share the same x-axis scale
)


# **Remove "street=" prefix from subplot titles**
fig.for_each_annotation(lambda ann: ann.update(text=ann.text.split("=")[-1]))

# Save & Show
fig.write_html("Selected_roads.html")
fig.show()



## Line graphs of daily traffic patterns by different weekdays:
This visualization shows the traffic patterns during the seven days of the week, for comparing different traffic patterns across different weekdays

In [6]:
# Group by Boro, quarter, and weekday
df_grouped = df.groupby(["weekday", "Boro", "quarter"], as_index=False)["Vol"].mean()

# Ensure weekdays are in the correct categorical order
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df_grouped["weekday"] = pd.Categorical(df_grouped["weekday"], categories=weekday_order, ordered=True)

# Start by plotting data for Monday
initial_weekday = "Monday"
df_initial = df_grouped[df_grouped["weekday"] == initial_weekday]

# Create the line plot using Plotly Express
fig = px.line(
    df_initial,
    x="quarter",
    y="Vol",
    color="Boro",
    labels={
        "Vol": "Average trafic",
        "quarter": "Quarter (0–95)",
        "Boro": "Borough"
    },
    title=f"Average trafic pr. quarter – {initial_weekday}"
)

# Create dropdown buttons for selecting different weekdays
dropdown_buttons = [
    {
        "label": day,
        "method": "update",
        "args": [
            {
                "x": [df_grouped[df_grouped["weekday"] == day]["quarter"]] * df_grouped["Boro"].nunique(),
                "y": [
                    df_grouped[(df_grouped["weekday"] == day) & (df_grouped["Boro"] == boro)]["Vol"]
                    for boro in df_grouped["Boro"].unique()
                ],
                "type": "scatter"
            },
            {"title": f"Average trafic pr. quarter – {day}"}
        ]
    }
    for day in weekday_order
]

# Update layout with dropdown menu
fig.update_layout(
    updatemenus=[
        {
            "buttons": dropdown_buttons,
            "direction": "down",
            "showactive": True,
            "x": 0.1,
            "y": 1.15,
            "xanchor": "left",
            "yanchor": "top"
        }
    ]
)

# Show the plot

fig.write_html("gennemsnitlig_trafik_ugedage")

fig.show()


## Heatmap over NYC's road network:
This visualization maps the intensity of traffic on individual roads, using color gradients (red/orange/yellow). It visually emphasizes which roads are the most heavily trafficked.

The code is not included in this notebook, because it is too heavy and exceeds 100MB in the notebook

## 6. Discussion

### What went well?
The data preprocessing and aggregation worked well, making it possible to extract clear patterns in weekday and hourly traffic.

The visualizations successfully captured different perspectives: temporal (time of day), spatial (boroughs, road segments), and directional flows.

The analysis provided intuitive explanations for traffic variations, such as remote work impact and borough-specific behaviors.

The combination of bar plots, line graphs, and maps gave a well-rounded overview of NYC traffic dynamics.

### What could be improved?
Granular temporal analysis over longer periods:
The current analysis is based on averaged values. Incorporating data from different weeks, months, or seasons could reveal seasonal or event-based fluctuations (e.g., holidays, weather impact).

Congestion and capacity analysis:
The data shows traffic volumes but does not directly indicate congestion levels or road capacity utilization. Including such metrics would provide a deeper understanding of traffic efficiency.

Predictive modeling:
Incorporating forecasting methods could help predict future traffic patterns based on historical data, which would be useful for city planning.

Demographic and behavioral data integration:
Combining traffic data with socioeconomic or commuting pattern datasets could give richer insights into why certain patterns emerge.

### Why improve these?
Traffic in NYC is a complex system influenced by many factors. Deeper, more granular analyses would give better tools for urban planning, congestion mitigation, and infrastructure improvements.

## 7. Contributions

As I (s215736) am the only member of the group, i have contributed to 100% of the assignment.